# Data Loaders

In [2]:
import os
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader

class PotholeDataset(Dataset):
    def __init__(self, image_dir, label_dir):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, image_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # normalize
        image = image.astype(np.float32) / 255.0
        print(image.shape)

        # create empty mask
        mask = np.zeros(image.shape[:-1], dtype=np.float32)

        label_path = os.path.join(self.label_dir, image_name.replace('.jpg', '.txt'))
        with open(label_path, 'r') as f:
            lines = f.readlines()
            for l in lines:
                parts = l.strip().split()
                class_id = int(parts[0])
                coords = np.array([float(x) for x in parts[1:]]).reshape(-1, 2)

                coords[:, 0] *= image.shape[0]
                coords[:, 1] *= image.shape[0]
                coords = coords.astype(np.int32)
                cv2.fillPoly(mask, [coords], color=1.0)
        mask = np.expand_dims(mask, axis=-1)           
        return image, mask

        

In [3]:
def getPotholeDataloader(batch_size, image_dir, label_dir):
    dataset = PotholeDataset(image_dir, label_dir)     
    return DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        drop_last=True, 
        num_workers=4
    )

In [4]:
# Test
dataset = PotholeDataset("/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/images", "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/labels")     
loader = getPotholeDataloader(4, 
                     "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/images", 
                     "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/labels")
dataset[0]

(640, 640, 3)


(array([[[0.16078432, 0.1764706 , 0.22352941],
         [0.16078432, 0.1764706 , 0.22352941],
         [0.16862746, 0.18431373, 0.23137255],
         ...,
         [0.5803922 , 0.5176471 , 0.46666667],
         [0.58431375, 0.52156866, 0.47058824],
         [0.5882353 , 0.5254902 , 0.4745098 ]],
 
        [[0.16470589, 0.18039216, 0.22745098],
         [0.16470589, 0.18039216, 0.22745098],
         [0.16470589, 0.18039216, 0.22745098],
         ...,
         [0.5803922 , 0.5176471 , 0.46666667],
         [0.58431375, 0.52156866, 0.47058824],
         [0.58431375, 0.52156866, 0.47058824]],
 
        [[0.17254902, 0.18431373, 0.24313726],
         [0.16862746, 0.18039216, 0.23921569],
         [0.16470589, 0.1764706 , 0.23529412],
         ...,
         [0.5764706 , 0.50980395, 0.47058824],
         [0.5803922 , 0.5137255 , 0.4745098 ],
         [0.5803922 , 0.5137255 , 0.4745098 ]],
 
        ...,
 
        [[0.36078432, 0.36862746, 0.3647059 ],
         [0.49019608, 0.49411765, 0.50980

# Rotary Position Embeddings (RoPE)

In [5]:
import jax
import jax.numpy as jnp 
import flax.linen as nn

def apply_rope(x, head_dim):
    # X -> B,S,H,head_dim
    seq_len = x.shape[1]
    num_heads = x.shape[2]
    dims = jnp.arange(seq_len, dtype=jnp.float32).reshape(-1, 1) # seq_len, 1
    i_2 = jnp.arange(0, head_dim, 2, dtype=jnp.float32)
    div_term = jnp.exp(-1* i_2 * jnp.log(100000)/head_dim) # head_dim//2
    freq = dims * div_term # seq_len, head_dim//2
    sin_freq = jnp.repeat(jnp.sin(freq), 2, axis=-1).reshape(1, seq_len, 1, head_dim)
    cos_freq = jnp.repeat(jnp.cos(freq), 2, axis=-1).reshape(1, seq_len, 1, head_dim)
    odd_terms = x[..., 1::2]
    even_terms = x[..., 0::2]
    x_tmp = jnp.stack([-odd_terms, even_terms], axis=-1).reshape(x.shape)
    return x * cos_freq + x_tmp * sin_freq


In [6]:
# test
key = jax.random.key(42)
apply_rope(jax.random.normal(key, shape=(5, 10, 12, 20)), 20)

Array([[[[-2.83046160e-02,  4.67131853e-01,  2.95702964e-01, ...,
           2.19057187e-01, -1.44850874e+00,  7.64187515e-01],
         [-2.41546974e-01, -1.17938101e+00, -1.93891835e+00, ...,
           2.27838308e-01, -5.57045221e-01, -8.83011103e-01],
         [-2.13500634e-01,  3.08011293e-01, -1.87219039e-01, ...,
          -4.09815162e-01,  3.67117763e-01,  1.18789649e+00],
         ...,
         [ 3.69938195e-01,  4.05289829e-01,  7.05405399e-02, ...,
           2.68640998e-03,  1.43296540e+00,  1.35599673e+00],
         [-1.62400186e+00, -1.72977507e+00, -1.44196999e+00, ...,
          -4.29603189e-01, -1.84735149e-01,  1.01921999e+00],
         [-8.93042564e-01,  4.46165234e-01,  8.14330697e-01, ...,
           1.07841909e-01,  2.07426429e-01, -8.38911682e-02]],

        [[ 3.85086596e-01,  7.22145557e-01, -7.14749396e-01, ...,
          -9.22494829e-02,  1.76541209e+00, -1.58433056e+00],
         [-1.95976257e+00,  3.71923447e-02,  1.21875060e+00, ...,
          -2.17437744e

# Multi-Head Self-Attention 

In [8]:
class MHSA(nn.Module):
    num_heads: int
    
    @nn.compact
    def __call__(self, x):
        # x -> Batch, Seq_len, dim
        b, s, dim = x.shape
        head_dim = dim//self.num_heads

        w_q = self.param('w_q', nn.initializers.lecun_normalizer(), (self.dim, self.dim))
        w_k = self.param('w_k', nn.initializers.lecun_normalizer(), (self.dim, self.dim))
        w_v = self.param('w_v', nn.initializers.lecun_normalizer(), (self.dim, self.dim))
        w_o = self.param('w_o', nn.initializers.lecun_normalizer(), (self.dim, self.dim))

        q = jnp.dot(x, w_q).reshape(b, s, self.num_heads, head_dim)
        k = jnp.dot(x, w_k).reshape(b, s, self.num_heads, head_dim)
        v = jnp.dot(x, w_v).reshape(b, s, self.num_heads, head_dim)

        q = apply_rope(q, head_dim)
        k = apply_rope(k, head_dim)
        
        qk_t = jnp.einsum("bqhd, bkhd -> bhqk", q, k) 
        atten_scores = qk_t/jnp.sqrt(head_dim)
        attention = jax.nn.softmax(atten_scores, axis=-1)
 
        att_out = jnp.einsum('bhqk, bkhd -> bqhd', attention, v).reshape(b, s, dim)
        
        out = jnp.dot(att_out, w_o)
        return out

# Transformer block and adaLN

In [9]:
class AdaLN(nn.Module):
    dim: int

    @nn.compact
    def __call__(self, x, cond):
        mlp_out = nn.Dense(self.out_dim*3, kernel_init=nn.initializers.zeros, bias_init=nn.initializers.zeros)(cond)
        shift, scale, gate = jnp.split(mlp_out, 3, axis=-1)
        x = nn.LayerNorm()(x)

        x = x * (1 + jnp.expand_dims(scale, axis = 1)) + jnp.expand_dims(shift, axis = 1)

        return x, jnp.expand_dims(gate, axis = 1)